##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [2]:
# -----------------------------
# ARTI 560 - Transfer Learning
# MobileNetV2 on CIFAR-10
# Optimized for VS Code CPU
# -----------------------------

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

print("GPU Available:", tf.config.list_physical_devices('GPU'))

# -----------------------------
# 1) Load CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

# -----------------------------
# 2) Data augmentation
# -----------------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

# -----------------------------
# 3) tf.data pipeline (SPEED UP)
# -----------------------------
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

val_size = int(0.1 * len(x_train))
x_val, y_val = x_train[:val_size], y_train[:val_size]
x_tr,  y_tr  = x_train[val_size:], y_train[val_size:]

train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr))\
    .shuffle(5000)\
    .batch(BATCH)\
    .cache()\
    .prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))\
    .batch(BATCH)\
    .cache()\
    .prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))\
    .batch(BATCH)\
    .cache()\
    .prefetch(AUTOTUNE)

# -----------------------------
# 4) Build MobileNetV2 backbone
#    Use 128 instead of 224 (FAST)
# -----------------------------
IMG_SIZE = 128

mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
mobilenet_base.trainable = False

# -----------------------------
# 5) Full Model
# -----------------------------
mobilenet_model = keras.Sequential([
    layers.Input(shape=(32,32,3)),
    data_augmentation,
    layers.Resizing(IMG_SIZE, IMG_SIZE),
    layers.Lambda(mobilenet_preprocess),
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(10)
])

mobilenet_model.summary()

# -----------------------------
# 6) Inspect architecture
# -----------------------------
print("\nTotal layers in MobileNetV2 backbone:", len(mobilenet_base.layers))

trainable_layers_mn = [l for l in mobilenet_base.layers if l.count_params() > 0]
print("Layers with learnable parameters (depth):", len(trainable_layers_mn))

print("\nSample learnable layers (first 10):")
for i, layer in enumerate(trainable_layers_mn[:10]):
    print(i, layer.name, layer.count_params())

print("\nSample learnable layers (last 10):")
start = max(0, len(trainable_layers_mn)-10)
for i, layer in enumerate(trainable_layers_mn[start:], start=start):
    print(i, layer.name, layer.count_params())

trainable_params = np.sum([np.prod(v.shape) for v in mobilenet_model.trainable_weights])
nontrainable_params = np.sum([np.prod(v.shape) for v in mobilenet_model.non_trainable_weights])

print("\nTrainable params:", trainable_params)
print("Non-trainable params:", nontrainable_params)

# -----------------------------
# 7) Compile + Train (Frozen)
# -----------------------------
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_mn = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    verbose=1
)

# -----------------------------
# 8) Test (Frozen)
# -----------------------------
test_loss_mn, test_acc_mn = mobilenet_model.evaluate(test_ds, verbose=0)
print("\nMobileNetV2 (frozen) test accuracy:", test_acc_mn)

# -----------------------------
# 9) Fine-tune last layers
# -----------------------------
mobilenet_base.trainable = True

for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

print("\nTrainable layers in backbone:",
      sum(l.trainable for l in mobilenet_base.layers),
      "/", len(mobilenet_base.layers))

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_mn_ft = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    verbose=1
)

# -----------------------------
# 10) Test (Fine-tuned)
# -----------------------------
test_loss_mn_ft, test_acc_mn_ft = mobilenet_model.evaluate(test_ds, verbose=0)
print("\nMobileNetV2 (fine-tuned) test accuracy:", test_acc_mn_ft)

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


Total layers in MobileNetV2 backbone: 154
Layers with learnable parameters (depth): 104

Sample learnable layers (first 10):
0 Conv1 864
1 bn_Conv1 128
2 expanded_conv_depthwise 288
3 expanded_conv_depthwise_BN 128
4 expanded_conv_project 512
5 expanded_conv_project_BN 64
6 block_1_expand 1536
7 block_1_expand_BN 384
8 block_1_depthwise 864
9 block_1_depthwise_BN 384

Sample learnable layers (last 10):
94 block_15_project 153600
95 block_15_project_BN 640
96 block_16_expand 153600
97 block_16_expand_BN 3840
98 block_16_depthwise 8640
99 block_16_depthwise_BN 3840
100 block_16_project 307200
101 block_16_project_BN 1280
102 Conv_1 409600
103 Conv_1_bn 5120

Trainable params: 12810
Non-trainable params: 2257984
704/704 ━━━━━━━━━━━━━━━━━━━━ 34s 42ms/step - accuracy: 0.5598 - loss: 1.2815 - val_accuracy: 0.8152 - val_loss: 0.5371

MobileNetV2 (frozen) test accuracy: 0.8119000196456909

Trainable layers in backbone: 30 / 154
704/704 ━━━━━━━━━━━━━━━━━━━━ 53s 63ms/step - accuracy: 0.6656 - l

1. Which model achieved the highest accuracy?

The fine-tuned ResNet50V2 achieved the highest test accuracy of 91.62%, outperforming the fine-tuned MobileNetV2 which achieved 83.15%.

2. Which model trained faster?

MobileNetV2 trained way faster than ResNet50V2.
MobileNetV2 required approximately 34–53 seconds per epoch, while ResNet50V2 required around 178–226 seconds per epoch.


3. How might the architecture explain the differences?

ResNet50V2 is a deeper network with residual connections that allow it to learn more complex feature representations, resulting in higher accuracy. In contrast, MobileNetV2 uses depthwise separable convolutions to reduce computational complexity, enabling faster training but with slightly lower accuracy.


